In [39]:
# =============================================================================
# CELL 1 — IMPORTS & CONFIGURATION (FISIK SPORT)
# =============================================================================

import pandas as pd
import numpy as np
import re
import math
import warnings
warnings.filterwarnings('ignore')

FILE_PATH = r'C:\Users\dadia\capstone-project\db\raw\C&F.xlsx'
BRAND_ID = 'f6385739-fd01-4e15-8cc0-ff2b5102c506'  # C&FUUID

SUPABASE_URL = 'https://zzfghscdyvwecxrjzqrn.supabase.co'
SUPABASE_KEY = 'sb_secret_cVHnlsiAHiZXaV1vjbl0bA_W_mtBGh_'

PLATFORM_MAP = {
    'tiktok' : '4d324e89-b6c0-44c2-8aac-f821f6087b9e',
    'shopee' : '045dbc37-9740-44cd-99b7-03489159173a',
    'multi'  : 'b4abffae-6146-4d40-b4bd-b70a00c2a4e7',
}

# ========== C&F Hosts → Staff IDs ==========
HOST_MAP = {
    # Main hosts (lowercase = PIERO-style, uppercase = original)
    'magdalena': 14, 'kayla': 12, 'aan': 28, 'sasha': 17,
    'lena': 13, 'vina': 23, 'mentari': 49, 'thania': 21,
    'wulan': 36, 'aliza': 50, 'michael': 51, 'sahra': 114,
    'inggrit': 10, 'takis': 19, 'arsi': 2, 'arsy': 2,
    'desya': 52, 'sopi': 55, 'amara': 87,
    'tata kristi': 53, 'tata': 64,
    
    # Newly added
    'rasya': 134, 'clara': 135, 'nazma': 136, 'caca': 137,
    'thara': 138, 'vonny': 139, 'falisha': 140, 'bintang': 141,
    'adam khodam': 142, 'eldo': 143, 'amara nath': 144, 'hb els': 145,
    
    # Dual hosts - map to first host
    'aliza&clara': 50,
    'aliza&rasya': 50,
    'aliza&thania': 50,
    'amara&aliza': 87,
    'amara&michael': 87,
    'amara, rasya': 87,
    'rasya cindy': 134,
    'rasya amara': 134,
    'rasya aliza': 134,
    'rasya&falisha': 134,
    'vonny&falisha': 139,
    'vonny&lena': 139,
    'wulan - rasya': 36,
    'wulan rasya': 36,
    'thania & michael': 21,
    'thania,vina': 21,
    'vina,thania': 23,
    'kayla*': 12,
    'wulan,rasya': 36,
    'thania ,vina': 21,
    'amara & michael': 87,
}


# Junk filters
JUNK_HOST_VALUES = {
    'host live', 'host', 'nama', 'name',
    'total', 'total sesi', 'total session',
    'off', 'off outing', 'libur', '', 'nan',
    'weekend deals', 'weekend\ndeals', 'lower price', 'lower\nprice',
    'higher price', 'higher\nprice', 
    'men scent session', 'men\nscent\nsession',
    'bukber vidhelp', 'internal', '-',
    'hb els bintang', 'hb els\nbintang',
    'hb els adam', 'hb els\nadam',
    'oppa calvin', 'hb els\noppa calvin',
    # Product names leaking from bottom of sheets:
    'police', 'brand', 'back up agency', 'back up pay agency',
    "beverly's secret", 'perfume republic', 'avicenna',
    'shirley may', 'adopt', 'pascal morabito', 'louis varel',
    'tous', 'mauboussin', 'halloween', 'marina de bourbon',
    'colour me', "perfumer's choice", 'sindy',
    'police to be my avatar', 'police rich edp 100 ml',
    'police to be exotic jungle', 'police to be born to shine',
    'police to be sweet girl edp 125 ml', 'police to be bad guy edt 125 ml',
    'police to be free to dare', 'police amber gold women edt 100 ml',
    'column 1', 'total sesi reg',
}
JUNK_PLATFORM_VALUES = {'platform', 'host', 'sesi', '', 'nan'}

print(f'✅ CELL 1 — C&F Configuration loaded')
print(f'   BRAND_ID: {BRAND_ID}')
print(f'   Hosts mapped: {len(HOST_MAP)}')

✅ CELL 1 — C&F Configuration loaded
   BRAND_ID: f6385739-fd01-4e15-8cc0-ff2b5102c506
   Hosts mapped: 54


In [40]:
# =============================================================================
# CELL 2 — HELPER FUNCTIONS
# =============================================================================

def find_header_row(df):
    """Find the row containing column headers."""
    for i in range(min(20, len(df))):
        row_values = [str(v).lower().strip() for v in list(df.iloc[i])]
        keywords = ['tanggal', 'jam', 'platform', 'host', 'revenue']
        matches = sum(1 for kw in keywords if any(kw in v for v in row_values))
        if matches >= 2:
            return i
    return None


def clean_date(val):
    """Convert various date formats to YYYY-MM-DD."""
    if pd.isna(val) or str(val).strip() in ('', 'nan', 'NaT'):
        return None
    if hasattr(val, 'strftime'):
        return val.strftime('%Y-%m-%d')
    
    s = str(val).strip()
    
    # Try ISO/pandas format first
    try:
        return pd.to_datetime(s).strftime('%Y-%m-%d')
    except:
        pass
    
    # Try Indonesian date format
    MONTH_MAP = {
        'januari':'01', 'februari':'02', 'maret':'03', 'april':'04',
        'mei':'05', 'juni':'06', 'juli':'07', 'agustus':'08',
        'september':'09', 'oktober':'10', 'november':'11', 'desember':'12',
        # English month names as fallback
        'january':'01', 'february':'02', 'march':'03', 'may':'05',
        'june':'06', 'july':'07', 'august':'08', 'october':'10',
        'december':'12'
    }
    s_lower = s.lower()
    for month_name, month_num in MONTH_MAP.items():
        if month_name in s_lower:
            parts = re.findall(r'\d+', s_lower)
            if len(parts) >= 2:
                day = parts[0].zfill(2)
                year = parts[-1]
                if len(year) == 4 and int(year) < 2000:
                    year = '20' + year[2:]
                elif len(year) == 2:
                    year = '20' + year
                try:
                    return pd.to_datetime(f'{year}-{month_num}-{day}').strftime('%Y-%m-%d')
                except:
                    pass
    return None


def clean_time(val):
    """Normalize time ranges to HH:MM-HH:MM."""
    if pd.isna(val) or str(val).strip() in ('', 'nan'):
        return None
    s = str(val).strip().replace('.', ':').replace(' ', '')
    if '-' in s:
        parts = s.split('-')
        return f"{parts[0].strip()}-{parts[1].strip()}"
    return s


def clean_revenue(val):
    """Convert revenue strings to integers."""
    if pd.isna(val):
        return 0
    s = str(val).strip()
    if s in ('', 'nan', '-', '0', 'Rp. 0'):
        return 0
    s = re.sub(r'[Rr][Pp][.\s]*', '', s)
    s = re.sub(r'[.\s]', '', s)
    s = s.replace(',', '.')
    try:
        return int(float(s))
    except:
        return 0


def clean_viewers(val):
    """Convert viewer/likes counts to integers."""
    if pd.isna(val):
        return 0
    s = str(val).strip().lower()
    if s in ('', 'nan', '-'):
        return 0
    if 'rb' in s:
        s = s.replace('rb', '').replace(',', '.').strip()
        try:
            return int(float(s) * 1000)
        except:
            pass
    if 'k' in s:
        s = s.replace('k', '').replace(',', '.').strip()
        try:
            return int(float(s) * 1000)
        except:
            pass
    s = re.sub(r'[,\s]', '', s)
    try:
        return int(float(s))
    except:
        return 0


def clean_host(val):
    """Clean host names."""
    if pd.isna(val):
        return None
    s = str(val).strip().lower()
    if s in JUNK_HOST_VALUES:
        return None
    if re.match(r'^\d+[\.,]?\d*$', s):
        return None
    s = re.sub(r'\s*\(.*?\)', '', s)
    s = re.sub(r'<br>', '', s)
    return s.strip()


def clean_platform(val):
    """Normalize platform names."""
    if pd.isna(val):
        return None
    s = str(val).strip().lower()
    if s in JUNK_PLATFORM_VALUES:
        return None
    if 'tiktok' in s or s == 'tt':
        return 'tiktok'
    if 'shopee' in s:
        return 'shopee'
    if 'multi' in s:
        return 'multi'
    return None


print('✅ CELL 2 — Helper functions defined')

✅ CELL 2 — Helper functions defined


In [41]:
# DEBUG CELL - Run this to find the error
import traceback

xl = pd.ExcelFile(FILE_PATH)
sheet_name = xl.sheet_names[0]  # Just first sheet
print(f'Testing: {sheet_name}')

match = re.search(r'(\d+)', sheet_name.upper().replace('PRIODE', 'PERIODE'))
p_num = int(match.group(1))

df_raw = pd.read_excel(FILE_PATH, sheet_name=sheet_name, header=None)
header_row = find_header_row(df_raw)
print(f'Header row: {header_row}')

raw_cols = [str(c).lower().strip() for c in df_raw.iloc[header_row]]
print(f'Columns: {raw_cols[:10]}')

data = df_raw.iloc[header_row + 1:].copy()
n_header = len(raw_cols)
n_data = data.shape[1]
if n_data > n_header:
    data = data.iloc[:, :n_header]

data.columns = raw_cols[:data.shape[1]]
print(f'Shape: {data.shape}')

# Try each step
try:
    data['date_clean'] = data.iloc[:, 0].apply(clean_date)
    print('✅ date ok')
except Exception as e:
    print(f'❌ date: {e}')

try:
    # Find host column
    host_col = None
    for c in data.columns:
        if 'host' in str(c).lower():
            host_col = c
            break
    print(f'Host column: {host_col}')
    data['host_clean'] = data[host_col].apply(clean_host)
    print('✅ host ok')
    print(f'Unique hosts: {data["host_clean"].value_counts().head(10)}')
except Exception as e:
    print(f'❌ host: {e}')
    traceback.print_exc()

Testing: PERIODE 1
Header row: 1
Columns: ['no', 'tanggal live', 'jam live', 'platform', 'host live', 'revenue shopee', 'viewers', 'comment', 'avg. view duration (detik)', 'penonton aktif']
Shape: (127, 30)
✅ date ok
Host column: host live
✅ host ok
Unique hosts: host_clean
aan          32
kayla        31
magdalena    30
sasha        30
lena          1
Name: count, dtype: int64


In [42]:
if 'time' not in df.columns:
    df['time'] = None

In [35]:
# =============================================================================
# CELL 3 — READ & CLEAN ALL SHEETS (FIXED FOR C&F DUAL COLUMNS)
# =============================================================================

def extract_sheet_data(df_raw, header_row, periode_num):
    """Extract and clean data from one sheet - handles dual SHOPEE/TIKTOK columns."""
    raw_cols = [str(c).lower().strip() for c in df_raw.iloc[header_row]]
    data = df_raw.iloc[header_row + 1:].copy()
    
    n_header = len(raw_cols)
    n_data = data.shape[1]
    if n_data > n_header:
        data = data.iloc[:, :n_header]
    
    # Use positional indexing (column numbers) instead of names
    col_map_position = {}
    
    for i, c in enumerate(raw_cols):
        s = str(c).lower().strip()
        
        # Date column
        if 'tanggal' in s and 'date' not in col_map_position.values():
            col_map_position[i] = 'date'
        # Time column
        elif s in ('jam live', 'jam', 'time') and 'time' not in col_map_position.values():
            col_map_position[i] = 'time'
        # Platform
        elif 'platform' in s and 'platform' not in col_map_position.values():
            col_map_position[i] = 'platform'
        # Host
        elif 'host' in s and 'host' not in col_map_position.values():
            col_map_position[i] = 'host'
        # Revenue TikTok (look for 'revenue tiktok' or 'rev' near 'tik')
        elif ('rev' in s and 'tik' in s) or s == 'revenue tiktok':
            col_map_position[i] = 'revenue_tiktok'
        # Revenue Shopee
        elif ('rev' in s and 'shop' in s) or s == 'revenue shopee':
            col_map_position[i] = 'revenue_shopee'
        # Viewers TikTok (second 'viewers')
        elif 'view' in s and 'viewers_tiktok' not in col_map_position.values():
            if 'viewers_tiktok' not in col_map_position.values():
                col_map_position[i] = 'viewers_shopee' if 'shop' in s else 'viewers_tiktok'
    
    # Build new DataFrame with mapped columns
    new_data = pd.DataFrame()
    for col_idx, col_name in col_map_position.items():
        if col_idx < data.shape[1]:
            new_data[col_name] = data.iloc[:, col_idx]
    
    # Add missing columns
    for col in ['date','time','platform','host',
                'revenue_tiktok','viewers_tiktok','likes_tiktok',
                'revenue_shopee','viewers_shopee','likes_shopee']:
        if col not in new_data.columns:
            new_data[col] = np.nan
    
    # Clean
    new_data['date'] = new_data['date'].apply(clean_date)
    new_data['time'] = new_data['time'].apply(clean_time)
    new_data['platform'] = new_data['platform'].apply(clean_platform)
    new_data['host'] = new_data['host'].apply(clean_host)
    
    for col in ['revenue_tiktok','revenue_shopee']:
        new_data[col] = new_data[col].apply(clean_revenue)
    for col in ['viewers_tiktok','viewers_shopee','likes_tiktok','likes_shopee']:
        new_data[col] = new_data[col].apply(clean_viewers)
    
    # Forward fill
    new_data['date'] = new_data['date'].ffill()
    new_data['platform'] = new_data['platform'].fillna('tiktok')
    
    # Filter
    new_data = new_data[new_data['host'].apply(lambda x: x is not None)]
    new_data = new_data[new_data['date'].notna()]
    
    new_data['periode_id'] = periode_num
    return new_data.reset_index(drop=True)


# ========== MAIN PROCESSING ==========
print('🔄 Reading C&F.xlsx...')
xl = pd.ExcelFile(FILE_PATH)
print(f'   Found {len(xl.sheet_names)} sheets')

all_frames = []

for sheet_name in xl.sheet_names:
    match = re.search(r'(\d+)', sheet_name.upper().replace('PRIODE', 'PERIODE'))
    if not match:
        continue
    p_num = int(match.group(1))
    
    try:
        df_raw = pd.read_excel(FILE_PATH, sheet_name=sheet_name, header=None)
        header_row = find_header_row(df_raw)
        if header_row is None:
            print(f'   ⚠️  Period {p_num}: No header')
            continue
        df_sheet = extract_sheet_data(df_raw, header_row, p_num)
        if len(df_sheet) > 0:
            all_frames.append(df_sheet)
            print(f'   ✅ Period {p_num:2d}: {len(df_sheet):3d} rows')
    except Exception as e:
        print(f'   ❌ Period {p_num}: {str(e)[:80]}')

if all_frames:
    df = pd.concat(all_frames, ignore_index=True)
    df = df.sort_values(['periode_id', 'date']).reset_index(drop=True)
    print(f'\n✅ CELL 3 — {len(df)} rows extracted')
else:
    print('\n❌ No data extracted!')
    df = pd.DataFrame()

🔄 Reading C&F.xlsx...
   Found 12 sheets
   ✅ Period  1: 127 rows
   ✅ Period  2: 161 rows
   ✅ Period  3: 199 rows
   ✅ Period  4: 233 rows
   ✅ Period  6: 379 rows
   ✅ Period  5: 288 rows
   ✅ Period  7: 383 rows
   ✅ Period  8: 378 rows
   ✅ Period  9: 444 rows
   ✅ Period 10: 571 rows
   ✅ Period 11: 454 rows
   ✅ Period 12: 461 rows

✅ CELL 3 — 4078 rows extracted


In [36]:
if 'time' not in df.columns:
    df['time'] = None

In [37]:
# =============================================================================
# CELL 3B — SPLIT DUAL HOSTS INTO SEPARATE ROWS
# =============================================================================

dual_hosts = {
    'aliza&clara': ['aliza', 'clara'],
    'aliza&rasya': ['aliza', 'rasya'],
    'aliza&thania': ['aliza', 'thania'],
    'amara&aliza': ['amara', 'aliza'],
    'amara&michael': ['amara', 'michael'],
    'amara, rasya': ['amara', 'rasya'],
    'rasya cindy': ['rasya', 'cindy'],
    'rasya amara': ['rasya', 'amara'],
    'rasya aliza': ['rasya', 'aliza'],
    'rasya&falisha': ['rasya', 'falisha'],
    'vonny&falisha': ['vonny', 'falisha'],
    'vonny&lena': ['vonny', 'lena'],
    'wulan - rasya': ['wulan', 'rasya'],
    'wulan rasya': ['wulan', 'rasya'],
    'thania & michael': ['thania', 'michael'],
    'thania,vina': ['thania', 'vina'],
    'vina,thania': ['vina', 'thania'],
    'thania ,vina': ['thania', 'vina'],
}

HOST_MAP['cindy'] = 4

new_rows = []
for idx, row in df.iterrows():
    host = str(row['host']).lower().strip()
    if host in dual_hosts:
        hosts = dual_hosts[host]
        n = len(hosts)
        for h in hosts:
            new_row = row.copy()
            new_row['host'] = h
            new_row['revenue_tiktok'] = int(row['revenue_tiktok'] / n)
            new_row['revenue_shopee'] = int(row['revenue_shopee'] / n)
            new_rows.append(new_row)
    else:
        new_rows.append(row)

df = pd.DataFrame(new_rows).reset_index(drop=True)
print(f'✅ Dual hosts split — {len(df)} total rows')

✅ Dual hosts split — 4113 total rows


In [43]:
# Fix NULL time and forward fill platform
df['platform'] = df['platform'].fillna('tiktok')
df['time'] = df['time'].fillna('00:00-00:00')

In [45]:
# =============================================================================
# CELL 4 — DATA QUALITY CHECK (C&F)
# =============================================================================

print('=' * 60)
print('DATA QUALITY REPORT — C&F')
print('=' * 60)

print(f'\n Total rows: {len(df)}')

print(f'\n Missing values:')
print(df.isnull().sum())

print(f'\n Platforms:')
print(df['platform'].value_counts(dropna=False))

print(f'\n Hosts (top 25):')
print(df['host'].value_counts().head(25))

print(f'\n Date range:')
print(f'   From: {df["date"].min()}')
print(f'   To:   {df["date"].max()}')

print(f'\n Rows per period:')
print(df['periode_id'].value_counts().sort_index())

# Check unmapped hosts
unmapped = df[~df['host'].isin(HOST_MAP.keys())]['host'].value_counts()
if len(unmapped) > 0:
    print(f'\n⚠️  UNMAPPED HOSTS (will get NULL host_id):')
    for host, count in unmapped.items():
        print(f'   {host}: {count} sessions')
else:
    print('\n✅ All hosts are mapped!')

print(f'\n Revenue TikTok:')
print(f'   Total: {df["revenue_tiktok"].sum():,.0f}')
print(f'   Mean:  {df["revenue_tiktok"].mean():,.0f}')
print(f'   Zeros: {(df["revenue_tiktok"] == 0).sum()} rows')

DATA QUALITY REPORT — C&F

 Total rows: 4113

 Missing values:
date                0
time                0
platform            0
host              518
revenue_shopee      0
viewers_tiktok      0
revenue_tiktok      0
likes_tiktok        0
viewers_shopee      0
likes_shopee        0
periode_id          0
dtype: int64

 Platforms:
platform
tiktok    4011
multi      102
Name: count, dtype: int64

 Hosts (top 25):
host
thania            662
kayla             469
aliza             441
wulan             347
rasya             311
vina              306
magdalena         288
lena              201
sasha             154
mentari           119
amara              66
clara              40
aan                38
amara nath         28
nazma              14
vonny               7
thara               6
adam khodam         5
caca                4
kayla*              4
police              4
back up agency      3
arsy                3
inggrit             3
brand               3
Name: count, dtype: int64

 Dat

In [48]:
# =============================================================================
# FIX: Forward fill host for merged cells 
# =============================================================================

# Convert None to NaN first, then ffill
df['host'] = df['host'].replace({None: np.nan})
df['host'] = df['host'].ffill()


print(f'Hosts after fix:')
print(df['host'].value_counts(dropna=False))
print(f'\nUnmapped hosts:')
unmapped = df[~df['host'].isin(HOST_MAP.keys())]['host'].value_counts()
print(unmapped if len(unmapped) > 0 else '✅ All mapped!')

Hosts after fix:
host
thania             832
kayla              493
aliza              474
rasya              419
wulan              386
vina               316
magdalena          288
lena               215
sasha              157
mentari            123
amara               66
inggrit             48
clara               45
aan                 41
amara nath          34
tata                33
adam khodam         15
nazma               14
vonny               11
thara                6
eldo                 6
kayla*               5
caca                 4
arsy                 3
bintang              3
sahra                2
falisha              2
amara & michael      2
sopi                 1
desya                1
takis                1
tata kristi          1
wulan,rasya          1
michael              1
Name: count, dtype: int64

Unmapped hosts:
✅ All mapped!


In [55]:
# =============================================================================
# CELL 5 — MAP TO SUPABASE SCHEMA
# =============================================================================
# Renames columns and converts types for Supabase insert.
# =============================================================================

if df.empty:
    print('❌ DataFrame is empty!')
else:
    df_insert = pd.DataFrame()
    
    # Map platform and host
    df_insert['platform_id'] = df['platform'].map(PLATFORM_MAP)
    df_insert['host_id'] = df['host'].map(HOST_MAP)
    
    # Direct mappings
    df_insert['date'] = df['date']
    df_insert['time'] = df['time']
    df_insert['brand_id'] = BRAND_ID
    df_insert['revenue_shopee'] = df['revenue_shopee']
    df_insert['viewers_shopee'] = df['viewers_shopee']
    df_insert['likes_shopee'] = df['likes_shopee']
    df_insert['revenue_tiktok'] = df['revenue_tiktok']
    df_insert['viewers_tiktok'] = df['viewers_tiktok']
    df_insert['likes_tiktok'] = df['likes_tiktok']
    df_insert['period_id'] = df['periode_id']
    
    # Convert to integers
    int_cols = ['revenue_shopee','viewers_shopee','likes_shopee',
                'revenue_tiktok','viewers_tiktok','likes_tiktok']
    for col in int_cols:
        df_insert[col] = df_insert[col].fillna(0).astype(int)
    
    df_insert['host_id'] = df_insert['host_id'].astype('Int64')
    
    print(f'✅ CELL 5 — Data mapped to schema')
    print(f'   Columns: {list(df_insert.columns)}')
    print(f'   Rows: {len(df_insert)}')
    print(f'\n   Sample:')
    print(df_insert.head(3).to_string())

✅ CELL 5 — Data mapped to schema
   Columns: ['platform_id', 'host_id', 'date', 'time', 'brand_id', 'revenue_shopee', 'viewers_shopee', 'likes_shopee', 'revenue_tiktok', 'viewers_tiktok', 'likes_tiktok', 'period_id']
   Rows: 4049

   Sample:
                            platform_id  host_id        date         time                              brand_id  revenue_shopee  viewers_shopee  likes_shopee  revenue_tiktok  viewers_tiktok  likes_tiktok  period_id
0  b4abffae-6146-4d40-b4bd-b70a00c2a4e7       14  2025-06-01  08:30-10:30  f6385739-fd01-4e15-8cc0-ff2b5102c506         6198336               0             0          145713            1157             0          1
1  b4abffae-6146-4d40-b4bd-b70a00c2a4e7       12  2025-06-01  11:30-13:30  f6385739-fd01-4e15-8cc0-ff2b5102c506         4157911               0             0          361080            1300             0          1
2  b4abffae-6146-4d40-b4bd-b70a00c2a4e7       28  2025-06-01  20:00-22:00  f6385739-fd01-4e15-8cc0-ff2b5102c506 

In [51]:
# =============================================================================
# CELL 6 — CONNECT TO SUPABASE
# =============================================================================

from supabase import create_client

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
print('✅ CELL 6 — Connected to Supabase')

✅ CELL 6 — Connected to Supabase


In [52]:
# =============================================================================
# CELL 7 — CHECK EXISTING DATA
# =============================================================================

check = supabase.table('live_sessions') \
    .select('id', count='exact') \
    .eq('brand_id', BRAND_ID) \
    .execute()

print(f'🔍 Found {check.count} existing rows for C&F in live_sessions')

if check.count > 0:
    print('   ⚠️  These will be deleted in CELL 7B')
else:
    print('   ✅ No existing data — safe to proceed to CELL 8')

🔍 Found 0 existing rows for C&F in live_sessions
   ✅ No existing data — safe to proceed to CELL 8


In [53]:
# =============================================================================
# CELL 8 — INSERT DATA
# =============================================================================

def replace_nan(val):
    if val is None:
        return None
    try:
        if pd.isna(val):
            return None
    except:
        pass
    if isinstance(val, (np.integer,)):
        return int(val)
    return val

records = []
for _, row in df_insert.iterrows():
    record = {col: replace_nan(row[col]) for col in df_insert.columns}
    records.append(record)

print(f'📤 Inserting {len(records)} rows...')

BATCH_SIZE = 200
success = 0
failed = 0

for i in range(0, len(records), BATCH_SIZE):
    batch = records[i:i + BATCH_SIZE]
    try:
        supabase.table('live_sessions').insert(batch).execute()
        success += len(batch)
        print(f'   ✅ Batch {i//BATCH_SIZE+1}: {len(batch)} rows (total: {success})')
    except Exception as e:
        failed += len(batch)
        print(f'   ❌ Batch {i//BATCH_SIZE+1} failed: {str(e)[:100]}')

print(f'\n✅ Success: {success} | ❌ Failed: {failed}')

📤 Inserting 4049 rows...
   ✅ Batch 1: 200 rows (total: 200)
   ✅ Batch 2: 200 rows (total: 400)
   ✅ Batch 3: 200 rows (total: 600)
   ✅ Batch 4: 200 rows (total: 800)
   ✅ Batch 5: 200 rows (total: 1000)
   ✅ Batch 6: 200 rows (total: 1200)
   ✅ Batch 7: 200 rows (total: 1400)
   ✅ Batch 8: 200 rows (total: 1600)
   ✅ Batch 9: 200 rows (total: 1800)
   ✅ Batch 10: 200 rows (total: 2000)
   ✅ Batch 11: 200 rows (total: 2200)
   ✅ Batch 12: 200 rows (total: 2400)
   ✅ Batch 13: 200 rows (total: 2600)
   ✅ Batch 14: 200 rows (total: 2800)
   ✅ Batch 15: 200 rows (total: 3000)
   ✅ Batch 16: 200 rows (total: 3200)
   ✅ Batch 17: 200 rows (total: 3400)
   ✅ Batch 18: 200 rows (total: 3600)
   ✅ Batch 19: 200 rows (total: 3800)
   ✅ Batch 20: 200 rows (total: 4000)
   ✅ Batch 21: 49 rows (total: 4049)

✅ Success: 4049 | ❌ Failed: 0


In [54]:
# =============================================================================
# CELL 9 — VERIFY
# =============================================================================

verify = supabase.table('live_sessions') \
    .select('period_id, revenue_shopee') \
    .eq('brand_id', BRAND_ID) \
    .execute()

if verify.data:
    vdf = pd.DataFrame(verify.data)
    summary = vdf.groupby('period_id').agg(
        sessions=('period_id','count'),
        total_rev=('revenue_shopee','sum')
    ).sort_index()
    print(f'✅ {len(vdf)} rows verified')
    print(summary.to_string())
else:
    print('❌ No data found!')

✅ 1000 rows verified
           sessions  total_rev
period_id                     
1               127  658120878
2               161          0
3               199          0
4               233          0
5               280          0
